# Phase 7 — Offline Evaluation
NDCG@K, MAP, Recall@K, and Precision@K implemented from scratch.
Full comparison across all models on the validation set.

In [ ]:
import sys, time
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data import load_movies
from src.cf import UserUserCF, ItemItemCF, MatrixFactorization
from src.content import TFIDFRecommender, SentenceTransformerRecommender
from src.hybrid import HybridRecommender
from src.evaluate import evaluate_model, ndcg_at_k, recall_at_k, average_precision

In [ ]:
train  = pd.read_csv('../data/train.csv')
val    = pd.read_csv('../data/val.csv')
movies = load_movies()
print(f'train: {train.shape}  val: {val.shape}')

## 1. Metric Sanity Check
Verify metrics on a toy example before running full evaluation.

In [ ]:
# Perfect recommendation: all top-3 are relevant
recs     = [1, 2, 3, 4, 5]
relevant = {1, 2, 3}

print('Perfect top-3 rec:')
print(f'  NDCG@3  = {ndcg_at_k(recs, relevant, 3):.4f}  (expect 1.0)')
print(f'  Recall@3 = {recall_at_k(recs, relevant, 3):.4f}  (expect 1.0)')
print(f'  MAP      = {average_precision(recs, relevant):.4f}  (expect 1.0)')

# Worst case: relevant items at the bottom
recs2 = [4, 5, 6, 1, 2, 3]
print()
print('Relevant items ranked 4th-6th:')
print(f'  NDCG@3  = {ndcg_at_k(recs2, relevant, 3):.4f}  (expect 0.0)')
print(f'  Recall@3 = {recall_at_k(recs2, relevant, 3):.4f}  (expect 0.0)')
print(f'  MAP      = {average_precision(recs2, relevant):.4f}  (expect 0.333)')

## 2. Fit All Models

In [ ]:
print('Fitting models...')

t0   = time.time()
uucf = UserUserCF(K=50).fit(train)
print(f'  UserUserCF:    {time.time()-t0:.1f}s')

t0   = time.time()
iicf = ItemItemCF(K=50).fit(train)
print(f'  ItemItemCF:    {time.time()-t0:.1f}s')

t0  = time.time()
mf  = MatrixFactorization(n_factors=64, n_epochs=2, lr=0.005, reg=0.1, batch_size=2048)
mf.fit(train, val)
print(f'  MatrixFact:    {time.time()-t0:.1f}s')

t0   = time.time()
tfidf = TFIDFRecommender(max_features=500).fit(movies, train)
print(f'  TF-IDF:        {time.time()-t0:.1f}s')

t0 = time.time()
st = SentenceTransformerRecommender().fit(movies, train)
print(f'  SentenceTrans: {time.time()-t0:.1f}s')

hybrid = HybridRecommender(iicf, st, alpha=1.0)  # best alpha from Phase 6
print('Done.')

## 3. Evaluate All Models at K=5 and K=10

In [ ]:
N_USERS = 200
K_LIST  = (5, 10)

models = {
    'User-User CF':         lambda uid: [m for m, _ in uucf.recommend(uid, n=10)],
    'Item-Item CF':         lambda uid: [m for m, _ in iicf.recommend(uid, n=10)],
    'Matrix Factorization': lambda uid: [m for m, _ in mf.recommend(uid, n=10)],
    'TF-IDF Content':       lambda uid: [m for m, _ in tfidf.recommend(uid, train, n=10)],
    'Sentence-Transformer': lambda uid: [m for m, _ in st.recommend(uid, train, n=10)],
    'Hybrid (α=1.0)':       lambda uid: [m for m, _ in hybrid.recommend(uid, train, n=10)],
}

results = {}
for name, fn in models.items():
    t0 = time.time()
    results[name] = evaluate_model(fn, val, k_list=K_LIST, n_users=N_USERS)
    print(f'{name:<25}  done in {time.time()-t0:.1f}s')

## 4. Results Table

In [ ]:
metrics = ['ndcg@5', 'ndcg@10', 'map', 'recall@5', 'recall@10', 'precision@10']
rows    = []
for name, res in results.items():
    rows.append({'Model': name, **{m: round(res.get(m, 0), 4) for m in metrics}})

df_results = pd.DataFrame(rows).set_index('Model')
df_results

In [ ]:
# Bar chart: NDCG@10 across models
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ['ndcg@10', 'map', 'recall@10']):
    vals  = df_results[metric].sort_values()
    colors = ['#d62728' if v == vals.max() else '#1f77b4' for v in vals]
    vals.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(metric)
    ax.set_xlabel('score')

plt.suptitle('Offline Evaluation — All Models', fontsize=13)
plt.tight_layout()
plt.savefig('../data/evaluation_results.png', dpi=100)
plt.show()
print('Saved to data/evaluation_results.png')

## 5. Summary

Full NDCG/MAP/Recall comparison across all 6 models on 200 val users.

**Key takeaways:**
- Item-Item CF dominates on dense MovieLens-1M across all ranking metrics
- NDCG penalizes bad ranking within the top-K more than Precision does
- MAP is the strictest metric — averages precision at every relevant item position
- Recall@K shows how much of the user's ground-truth is captured; CF recovers far more than content models